In [ ]:
!pip install squarify


In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import squarify
sys.path.append('..')
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('future.no_silent_downcasting', True)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from scipy.stats import mannwhitneyu, ttest_ind

import utilities.functions as functions

from utilities.graficos import plot_metricas
from utilities.graficos import (
  boxplot_meses
 
)


from utilities.functions import (
    gerar_stats,
    pedidos_group,
    criacao_ordens,
    conversao_imediata,
   
)

from utilities.testes_estatisticos import testes,teste_proporcao_por_janela
from utilities.outliers import(
    outlier_method,
    mark_outliers_iqr_zscore_mad
)

In [ ]:
from pathlib import Path
BASE_PATH = Path("/Users/maceli/ifood_cs/dados") 
df = pd.read_parquet(BASE_PATH / "gold" / "df_publico.parquet")

In [ ]:
df=df[df['order_created_month']==12]#.head(1000)
df.head()

In [ ]:
outlier_method(df,var='order_total_amount')

In [ ]:
df = mark_outliers_iqr_zscore_mad(df)


In [ ]:
df_outliers = df[
   # df['outlier_iqr'] & 
   # df['outlier_zscore'] & 
    df['outlier_mad']]
print(len(df_outliers))
df_outliers.describe()

In [ ]:
df=df[~df['customer_id'].isin(df_outliers['customer_id'].unique())]

In [ ]:
df['outlier_mad'].unique()

In [ ]:
data_maxima = df['order_created_at'].max()


In [ ]:
#df=df[df['customer_id']!='361e229dbc1b985e1aacb3e70384782a05d77ad6db53e7e511fe2147ee09a890']

In [ ]:
rfm = df.groupby(['customer_id','is_target']).agg({
    'order_created_at': lambda x: (data_maxima - x.max()).days,  
    'unique_order_hash': 'count',  
    'order_total_amount': 'sum'  
}).reset_index()

In [ ]:
rfm.head()

In [ ]:

rfm.columns = ['customer_id','is_target', 'recencia', 'frequencia', 'valor_monetario']

rfm.head()

In [ ]:
rfm.shape

In [ ]:

for col, inv in [('recencia', True), ('frequencia', False), ('valor_monetario', False)]:
    rank = rfm[col].rank(method='first')
    if inv:
        rfm[f'{col[0]}_score'] = pd.qcut(rank, q=4, labels=[4,3,2,1])
    else:
        rfm[f'{col[0]}_score'] = pd.qcut(rank, q=4, labels=[1,2,3,4])

rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['v_score'].astype(str)



In [ ]:
rfm.reset_index().sort_values(by=['f_score'], ascending=[False]).head()


In [ ]:
df_stats_mes = (
        rfm.groupby(['rfm_score','is_target'])
          .agg(
              total_clientes=('customer_id', 'nunique'),
              media=("valor_monetario", 'mean'))) .reset_index().sort_values(by=['media'], ascending=[False])

df_stats_mes.head()

In [ ]:
rfm_avg = rfm.pivot_table(
    index='r_score', 
    columns='f_score', 
    values='valor_monetario', 
    aggfunc='mean'
)


In [ ]:
# Média do Valor Monetário por célula RFM

plt.figure(figsize=(10, 6))
sns.heatmap(rfm_avg, annot=True, fmt='.0f', cmap='YlOrRd', cbar_kws={'label': 'Valor Médio (R$)'})
plt.title('Valor Médio por Segmento RFM')
plt.xlabel('F Score (Frequência)')
plt.ylabel('R Score (Recência)')
plt.show()


In [ ]:

rfm_treemap = rfm_avg.reset_index().melt(
    id_vars='r_score',
    var_name='f_score',
    value_name='valor_medio'
)

In [ ]:
rfm_treemap = rfm_avg.reset_index().melt(
    id_vars='r_score',
    var_name='f_score',
    value_name='valor_medio'
)

# 2) Ordenar por R e depois por F
rfm_treemap = rfm_treemap.sort_values(['r_score', 'f_score']).reset_index(drop=True)

# 3) Criar rótulos
rfm_treemap["label"] = (
    "R:" + rfm_treemap["r_score"].astype(str) +
    " | F:" + rfm_treemap["f_score"].astype(str) +
    "\nR$ " + rfm_treemap["valor_medio"].round(0).astype(str)
)

# 4) Normalizar valores para o colormap
norm_vals = rfm_treemap["valor_medio"] / rfm_treemap["valor_medio"].max()
colors = plt.cm.RdYlGn(norm_vals)   # verde = alto valor, vermelho = baixo


# 5) Plot do Treemap
plt.figure(figsize=(12, 7))

squarify.plot(
    sizes=rfm_treemap["valor_medio"],
    label=rfm_treemap["label"],
    color=colors,
    alpha=0.9,
    text_kwargs={'fontsize': 10, 'color': 'black'}
)

plt.title("Treemap RFM — Valor Médio por Segmento (R x F)", fontsize=15)
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
seg_map = (
    rfm_avg
    .reset_index()
    .melt(id_vars='r_score', var_name='f_score', value_name='media_valor')
)

# 2) rotula cada média como "media X"
seg_map['segmento'] = 'media ' + seg_map['media_valor'].round(0).astype(int).astype(str)

# 3) agrupa 46, 47 e 48 num único grupo
seg_map.loc[seg_map['media_valor'].round().isin([43, 44, 42]), 'segmento'] = 'media 42-44'

# 4) aplica no dataframe original
rfm_segmentado = rfm.merge(
    seg_map[['r_score', 'f_score', 'segmento']],
    on=['r_score', 'f_score'],
    how='left'
)

In [ ]:
rfm_segmentado.head()

In [ ]:
boxplot_meses(rfm_segmentado,var_cat='segmento',var_cont='valor_monetario')